In [22]:
# Knowledge base: IF all conditions are true THEN conclusion
rules = [
    {"id": "R1", "conditions": ["has_hair", "gives_milk"], "conclusion": "mammal"},
    {"id": "R2", "conditions": ["has_feathers"], "conclusion": "bird"},
    {"id": "R3", "conditions": ["mammal", "eats_meat"], "conclusion": "carnivore"},
    {"id": "R4", "conditions": ["mammal", "has_tawny_color", "has_black_stripes"], "conclusion": "tiger"},
    {"id": "R5",
 "conditions": ["long_neck", "long_legs"],
 "conclusion": "giraffe"},
    {"id": "R6", "conditions": ["has_black_stripes", "not_carnivore"], "conclusion": "zebra"},
]

animals = ["tiger", "giraffe", "zebra"]

In [23]:
def forward_chaining(initial_facts, rules):
    working_memory = set(initial_facts)
    firing_sequence = []

    while True:
        conflict_set = []

        # Find all rules that can currently fire
        for rule in rules:
            conditions_satisfied = True

            for condition in rule["conditions"]:
                if condition not in working_memory:
                    conditions_satisfied = False
                    break

            if conditions_satisfied and rule["conclusion"] not in working_memory:
                conflict_set.append(rule)

        # Stop if no rule can fire
        if len(conflict_set) == 0:
            break

        # Select the first rule according to rule order
        selected_rule = conflict_set[0]

        # Fire the rule
        working_memory.add(selected_rule["conclusion"])
        firing_sequence.append(selected_rule["id"])

    identified_animal = None
    for animal in animals:
        if animal in working_memory:
            identified_animal = animal
            break

    return working_memory, firing_sequence, identified_animal

## Example: Tiger Identification

Initial facts:
- `has_hair`
- `gives_milk`
- `eats_meat`
- `has_tawny_color`
- `has_black_stripes`

Expected reasoning:
`has_hair + gives_milk → mammal` → `mammal + eats_meat → carnivore` → `mammal + tawny_color + black_stripes → tiger`.

In [24]:
tiger_facts = {
    "has_hair",
    "gives_milk",
    "eats_meat",
    "has_tawny_color",
    "has_black_stripes"
}

wm, sequence, animal = forward_chaining(tiger_facts, rules)

print("Final Working Memory:")
print(sorted(wm))
print("\nFiring Sequence:")
print(" -> ".join(sequence))
print("\nIdentified Animal:")
print(animal if animal else "No identification")

Final Working Memory:
['carnivore', 'eats_meat', 'gives_milk', 'has_black_stripes', 'has_hair', 'has_tawny_color', 'mammal', 'tiger']

Firing Sequence:
R1 -> R3 -> R4

Identified Animal:
tiger


## Exercise 1 – Identify Tiger, Giraffe and Zebra

The assignment asks us to add rules and facts for tiger, giraffe and zebra. Giraffe uses long neck and long legs, while zebra uses black stripes and not-carnivore information.

In [25]:
test_cases = {
    "Tiger": {
        "has_hair", "gives_milk", "eats_meat",
        "has_tawny_color", "has_black_stripes"
    },
    "Giraffe": {
        "long_neck", "long_legs"
    },
    "Zebra": {
        "has_black_stripes", "not_carnivore"
    }
}

for name, facts in test_cases.items():
    wm, sequence, animal = forward_chaining(facts, rules)
    print("Initial Facts:", sorted(facts))
    print("Final Working Memory:", sorted(wm))
    print("Firing Sequence:", " -> ".join(sequence) if sequence else "None")
    print("Identified Animal:", animal if animal else "No identification")
    print()

Initial Facts: ['eats_meat', 'gives_milk', 'has_black_stripes', 'has_hair', 'has_tawny_color']
Final Working Memory: ['carnivore', 'eats_meat', 'gives_milk', 'has_black_stripes', 'has_hair', 'has_tawny_color', 'mammal', 'tiger']
Firing Sequence: R1 -> R3 -> R4
Identified Animal: tiger

Initial Facts: ['long_legs', 'long_neck']
Final Working Memory: ['giraffe', 'long_legs', 'long_neck']
Firing Sequence: R5
Identified Animal: giraffe

Initial Facts: ['has_black_stripes', 'not_carnivore']
Final Working Memory: ['has_black_stripes', 'not_carnivore', 'zebra']
Firing Sequence: R6
Identified Animal: zebra



## Exercise 2 – No Identification

Give facts that do not allow any specific animal to be identified. The engine should report the most specific class it can establish.

In [26]:
unknown_facts = {"has_hair", "gives_milk"}

wm, sequence, animal = forward_chaining(unknown_facts, rules)

print("Initial Facts:", sorted(unknown_facts))
print("Final Working Memory:", sorted(wm))
print("Firing Sequence:", " -> ".join(sequence) if sequence else "None")

if animal is None:
    classes = ["mammal", "bird", "carnivore"]
    established = []
    for item in classes:
        if item in wm:
            established.append(item)

    if len(established) > 0:
        print("Identified Animal: No specific animal")
        print("Most specific class established:", established[-1])
    else:
        print("Identified Animal: No identification")
        print("Most specific class established: None")
else:
    print("Identified Animal:", animal)

Initial Facts: ['gives_milk', 'has_hair']
Final Working Memory: ['gives_milk', 'has_hair', 'mammal']
Firing Sequence: R1
Identified Animal: No specific animal
Most specific class established: mammal


## Exercise 3 – Fire ALL Applicable Rules Per Cycle

In the original engine, only the first applicable rule is fired in each cycle. In this modified version, every applicable rule is fired in the same cycle. This can produce a different firing sequence.

In [27]:
def forward_chaining_all_rules(initial_facts, rules):
    working_memory = set(initial_facts)
    firing_sequence = []

    while True:
        conflict_set = []

        for rule in rules:
            conditions_satisfied = True

            for condition in rule["conditions"]:
                if condition not in working_memory:
                    conditions_satisfied = False
                    break

            if conditions_satisfied and rule["conclusion"] not in working_memory:
                conflict_set.append(rule)

        if len(conflict_set) == 0:
            break

        # Fire every currently applicable rule
        new_fact_added = False
        for rule in conflict_set:
            if rule["conclusion"] not in working_memory:
                working_memory.add(rule["conclusion"])
                firing_sequence.append(rule["id"])
                new_fact_added = True

        if not new_fact_added:
            break

    identified_animal = None
    for animal in animals:
        if animal in working_memory:
            identified_animal = animal
            break

    return working_memory, firing_sequence, identified_animal

In [28]:
print("FIRST-RULE STRATEGY")
wm1, seq1, animal1 = forward_chaining(tiger_facts, rules)
print("Firing Sequence:", " -> ".join(seq1))
print("Animal:", animal1)

print("\nALL-RULES STRATEGY")
wm2, seq2, animal2 = forward_chaining_all_rules(tiger_facts, rules)
print("Firing Sequence:", " -> ".join(seq2))
print("Animal:", animal2)



FIRST-RULE STRATEGY
Firing Sequence: R1 -> R3 -> R4
Animal: tiger

ALL-RULES STRATEGY
Firing Sequence: R1 -> R3 -> R4
Animal: tiger
